# **Notebook 3: Baseline Model Evaluation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] Internet access (to download the model)
- [ ] GPU runtime enabled (Runtime → Change runtime type → T4 GPU)

**Files this notebook will CREATE:**
- [ ] `outputs.json` — `test_query`, `ground_truth`, `baseline_output` _(Required by NB4, NB5, NB7)_

---

## **Stage 3: Solution V1 (Retrieval-Assisted Generation)**

### **Task 3.1: Establish Baseline Performance**

#### **3.1.1 Execute Baseline Inference [2 marks]**
**The Task:** Load the pre-trained base model in 4-bit quantization and generate a response to an ambiguous shipping-delay query without any context.

**Hints & Tips:**
* Use `do_sample=False` for deterministic output. Do NOT pair `temperature=0.0` with `do_sample=False` — it throws a deprecation warning. Use `temperature=None, top_p=None`.
* `BitsAndBytesConfig(load_in_4bit=True)` shrinks the 1.5B model to ~750MB VRAM.
* `max_new_tokens=120` gives room for a complete answer.

**Model Selection:**
* **Qwen/Qwen2.5-1.5B-Instruct** (recommended) — must match what you used in NB2.
* **TinyLlama-1.1B-Chat** — lighter, weaker structured output.
* **Llama-3-8B-Instruct** — best quality, may OOM on free T4 during fine-tuning.

**Learner Inference:** This establishes your zero-shot baseline. Every later improvement is measured against this exact output.

In [7]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


In [8]:
!pip install -q -U bitsandbytes accelerate transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 127.4 MB/s eta 0:00:00


In [9]:
import bitsandbytes as bnb
print(bnb.__version__)

0.50.0


In [10]:
!pip uninstall -y bitsandbytes
!pip install -U bitsandbytes>=0.46.1 accelerate transformers

Found existing installation: bitsandbytes 0.50.0
Uninstalling bitsandbytes-0.50.0:
  Successfully uninstalled bitsandbytes-0.50.0


In [15]:
!pip install "bitsandbytes>=0.46.1"

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))

CUDA available: True
GPU Name: Tesla T4


In [2]:
# YOUR CODE HERE
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Configuration Parameters
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

# Define the ambiguous shipping delay query and SOP ground truth
test_query = "My order is delayed. When will it arrive and can I get a refund for the delay?"
ground_truth = "Domestic orders deliver within 3-7 business days. Expedited shipping guarantees delivery in 2 business days; if delayed, shipping costs are refunded."

print(f"Loading base model: {MODEL_ID}...")

# 2. Configure 4-Bit Quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# 3. Load Tokenizer and Base Model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto"
)

# 4. Format Prompt using Chat Template
messages = [
    {"role": "system", "content": "You are a helpful customer support assistant."},
    {"role": "user", "content": test_query}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# 5. Execute Deterministic Inference
with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        temperature=None,
        top_p=None
    )

# Extract only the newly generated response tokens
generated_tokens = outputs[0][inputs['input_ids'].shape[1]:]
baseline_output = tokenizer.decode(generated_tokens, skip_special_tokens=True)

# 6. Display Baseline Results
print("\n--- Baseline Inference Result ---")
print(f"Test Query      : {test_query}")
print(f"Ground Truth    : {ground_truth}")
print(f"\nBaseline Output :\n{baseline_output}")

Loading base model: Qwen/Qwen2.5-1.5B-Instruct...


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]


--- Baseline Inference Result ---
Test Query      : My order is delayed. When will it arrive and can I get a refund for the delay?
Ground Truth    : Domestic orders deliver within 3-7 business days. Expedited shipping guarantees delivery in 2 business days; if delayed, shipping costs are refunded.

Baseline Output :
I'm sorry to hear that your order is delayed. We apologize for any inconvenience this may cause you.

We understand that delays in delivery can be frustrating, but we want to assure you that we're doing everything possible to ensure timely shipping. Our team works hard every day to keep orders on track, and we appreciate your understanding during this time.

As for refunds, unfortunately, we cannot provide a refund for delays in delivery at this time. However, if there were any issues with the product itself or if you need to return an item due to a defect or other reasons, please contact our customer service


In [13]:
!pip show bitsandbytes

Name: bitsandbytes
Version: 0.50.0
Summary: k-bit optimizers and matrix multiplication routines.
Home-page: https://github.com/bitsandbytes-foundation/bitsandbytes
Author: 
Author-email: Tim Dettmers <dettmers@cs.washington.edu>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: numpy, packaging, torch
Required-by: 


#### **3.1.2 Evaluate Baseline Quality [2 marks]**
**The Task:** Assess the baseline output for factual inaccuracies against the ground-truth SOP rule.

**Hints & Tips:**
* Compare against the known rule: "Domestic orders deliver within 3-7 business days."
* Did the model invent a timeline? Mention a non-existent tracking system or department?
* Document every hallucination — it justifies Stages 3 and 4.

**Learner Inference:** This hallucination is exactly why you build Stage 3 (a database) and Stage 4 (a router).

In [3]:
# Task 3.1.2: Evaluate Baseline Performance & Document Hallucinations

def evaluate_baseline(query, ground_truth_sop, output_text):
    print("=" * 70)
    print("TASK 3.1.2: BASELINE EVALUATION & HALLUCINATION ANALYSIS")
    print("=" * 70)

    print(f"\n[USER QUERY]\n{query}")
    print(f"\n[GROUND TRUTH SOP RULE]\n{ground_truth_sop}")
    print(f"\n[BASELINE MODEL OUTPUT]\n{output_text}")
    print("-" * 70)

    # Quantitative comparison / Metrics breakdown
    sop_keywords = ["3-7 business days", "2 business days", "expedited shipping"]
    found_sop_facts = [kw for kw in sop_keywords if kw.lower() in output_text.lower()]

    print("\n[FACTUAL ACCURACY METRICS]")
    print(f"SOP Policy Compliance : {len(found_sop_facts)} / {len(sop_keywords)} key constraints met")
    print(f"Contains Ground Truth : {'YES' if len(found_sop_facts) > 0 else 'NO (Completely Unanchored)'}")
    print("-" * 70)

# Execute evaluation with variables defined in 3.1.1
evaluate_baseline(test_query, ground_truth, baseline_output)

TASK 3.1.2: BASELINE EVALUATION & HALLUCINATION ANALYSIS

[USER QUERY]
My order is delayed. When will it arrive and can I get a refund for the delay?

[GROUND TRUTH SOP RULE]
Domestic orders deliver within 3-7 business days. Expedited shipping guarantees delivery in 2 business days; if delayed, shipping costs are refunded.

[BASELINE MODEL OUTPUT]
I'm sorry to hear that your order is delayed. We apologize for any inconvenience this may cause you.

We understand that delays in delivery can be frustrating, but we want to assure you that we're doing everything possible to ensure timely shipping. Our team works hard every day to keep orders on track, and we appreciate your understanding during this time.

As for refunds, unfortunately, we cannot provide a refund for delays in delivery at this time. However, if there were any issues with the product itself or if you need to return an item due to a defect or other reasons, please contact our customer service
---------------------------------

---
## Save Artifacts for Downstream Notebooks

**IMPORTANT:** Saves the baseline output. Notebooks 4, 5, and 7 depend on this file.

In [4]:
# YOUR CODE HERE
import json

# Compile outputs from Task 3.1.1
baseline_data = {
    "test_query": test_query,
    "ground_truth": ground_truth,
    "baseline_output": baseline_output
}

# Save to outputs.json
output_filename = "outputs.json"
with open(output_filename, "w", encoding="utf-8") as f:
    json.dump(baseline_data, f, indent=4)

print(f"Successfully created {output_filename}:")
print(json.dumps(baseline_data, indent=2))

Successfully created outputs.json:
{
  "test_query": "My order is delayed. When will it arrive and can I get a refund for the delay?",
  "ground_truth": "Domestic orders deliver within 3-7 business days. Expedited shipping guarantees delivery in 2 business days; if delayed, shipping costs are refunded.",
  "baseline_output": "I'm sorry to hear that your order is delayed. We apologize for any inconvenience this may cause you.\n\nWe understand that delays in delivery can be frustrating, but we want to assure you that we're doing everything possible to ensure timely shipping. Our team works hard every day to keep orders on track, and we appreciate your understanding during this time.\n\nAs for refunds, unfortunately, we cannot provide a refund for delays in delivery at this time. However, if there were any issues with the product itself or if you need to return an item due to a defect or other reasons, please contact our customer service"
}


In [5]:
import os

# Get full absolute path
file_path = os.path.abspath("outputs.json")
print(f"File location: {file_path}")

# Verify file existence and content
if os.path.exists(file_path):
    print("\n--- File Contents ---")
    with open(file_path, "r") as f:
        print(f.read())
else:
    print("File not found! Make sure you executed the saving cell.")

File location: /content/outputs.json

--- File Contents ---
{
    "test_query": "My order is delayed. When will it arrive and can I get a refund for the delay?",
    "ground_truth": "Domestic orders deliver within 3-7 business days. Expedited shipping guarantees delivery in 2 business days; if delayed, shipping costs are refunded.",
    "baseline_output": "I'm sorry to hear that your order is delayed. We apologize for any inconvenience this may cause you.\n\nWe understand that delays in delivery can be frustrating, but we want to assure you that we're doing everything possible to ensure timely shipping. Our team works hard every day to keep orders on track, and we appreciate your understanding during this time.\n\nAs for refunds, unfortunately, we cannot provide a refund for delays in delivery at this time. However, if there were any issues with the product itself or if you need to return an item due to a defect or other reasons, please contact our customer service"
}


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 4.**

- [ ] Base model loaded in 4-bit without errors
- [ ] Baseline output generated for `test_query`
- [ ] Hallucination assessment documented
- [ ] **`outputs.json` saved** with `test_query`, `ground_truth`, `baseline_output` ← _CRITICAL for NB4, 5, 7_
- [ ] GPU runtime enabled

**If any item is unchecked, fix it before moving on.**